### Data loading, package installation

In [138]:
# load the csv file in "data" folder called raw_joined_data.csv
data <- read.csv("data/raw_joined_data.csv")

# install and load dplyr and purrr packages if not installed
if (!require(dplyr)) install.packages("dplyr")
if (!require(purrr)) install.packages("purrr")
if (!require(readr)) install.packages("readr")
if (!require(zoo)) install.packages("zoo")
if (!require(lubridate)) install.packages("lubridate")
library(dplyr)
library(purrr)
library(readr)
library(zoo)
library(lubridate)

### Debate_id processing

In [139]:
# Remove some inconsistencies
data <- data[!(data$speaker_first_debate_date == "2024-11-08" & data$speaker_name == "Šmejkal Jakub"), ]
data <- data[!(data$debate_id == "11054"), ]

### Category processing

In [140]:
# Combine the columns and find all unique, non-empty values
unique_categories <- unique(c(data$category_1, data$category_2, data$category_3))
unique_categories <- unique_categories[unique_categories != "" & !is.na(unique_categories)]

# View the result
print(unique_categories)

# Add these columns to your dataframe and initialize with 0
data[unique_categories] <- 0

 [1] "Environment / Climate"    "Social Justice"          
 [3] "Economics"                "Philosophy"              
 [5] "War / Conflict"           "Arts / Entertainment"    
 [7] "International Relations"  "Psychology"              
 [9] "History"                  "Politics"                
[11] "Social Media"             "Science / Technology"    
[13] "Family / Parenting"       "Urban Planning / Housing"
[15] "Law"                      "Sports"                  
[17] "Religion"                 "Work / Labor"            
[19] "Culture"                  "Media / Journalism"      


In [141]:
# Create a list of the 3 pairs of columns to iterate through
cat_cols <- c("category_1", "category_2", "category_3")
score_cols <- c("category_1_score", "category_2_score", "category_3_score")

# Loop through each row and each category/score pair
for (i in 1:nrow(data)) {
  for (j in 1:3) {
    category_val <- data[i, cat_cols[j]]
    score_val <- data[i, score_cols[j]]
    
    # Check if the category is not empty or NA, and the column exists
    if (!is.na(category_val) && category_val != "" && category_val %in% colnames(data)) {
      # Add the score to the existing value in that category's column
      data[i, category_val] <- score_val
    }
  }
}

# Remove the original category and score columns
data <- data[, !(names(data) %in% c(cat_cols, score_cols))]

### Affirmative side

In [142]:
# Rename column "side" to "aff_side"
data <- data %>% rename(aff_side = side)
# Convert aff_side to logical: "aff" -> TRUE, "neg" -> FALSE
data$aff_side <- ifelse(data$aff_side == "aff", TRUE, FALSE)

### Speaker position

In [143]:
# Speaker position to one-hot encoding
data <- data %>%
       mutate(speaker_position_1 = ifelse(speaker_position == 1, TRUE, FALSE),
              speaker_position_2 = ifelse(speaker_position == 2, TRUE, FALSE)) %>%
       # Remove the original column
       select(-speaker_position)

### Add "rolling_avg_5" (average of last 5 debates)

In [144]:
# Calculate historic average and recent form as doubles (floats)
data <- data %>%
  group_by(speaker_name) %>%
  mutate(
    # Historic average points per debate
    rolling_avg_5 = as.numeric(rollapplyr(
      lag(speaker_points), 
      width = 5, 
      FUN = mean, 
      na.rm = TRUE, 
      # Handle edge cases where less than 5 previous debates exist
      fill = NA,
      partial = TRUE
    ))
  ) %>%
  ungroup()

### Add "avg_teammate_rolling_score" and "avg_opponent_rolling_score"

In [145]:
data <- data %>%
  group_by(debate_id) %>%
  mutate(
    # Use map_dbl to allow decimal (float) output
    avg_teammate_rolling_score = map_dbl(row_number(), function(i) {
      current_is_aff <- aff_side[i]
      others_on_side <- rolling_avg_5[aff_side == current_is_aff & row_number() != i]
      
      # Calculate mean and return as is (mean returns a double by default)
      res <- mean(others_on_side, na.rm = TRUE)
      if (is.nan(res)) return(NA_real_)
      return(res) 
    }),
    
    avg_opponent_rolling_score = map_dbl(row_number(), function(i) {
      current_is_aff <- aff_side[i]
      opponents <- rolling_avg_5[aff_side != current_is_aff]
      
      res <- mean(opponents, na.rm = TRUE)
      if (is.nan(res)) return(NA_real_)
      return(res)
    })
  ) %>%
  ungroup()

### Add "cumulative_debates"

In [146]:
# Calculate the number of debates each player had before the current one
data <- data %>%
  group_by(speaker_name) %>%
  mutate(cumulative_debates = row_number() - 1) %>%
  ungroup()

### Add "years_debating"
- if first debate in 2024 -> years_debating = 1
- if first debate in 2023 -> years_debating = 2
- ...

In [147]:
data <- data %>%
  mutate(
    # Ensure the column is recognized as a Date object
    speaker_first_debate_date = as.Date(speaker_first_debate_date),
    # Apply the year logic
    years_debating = ifelse(is.na(speaker_first_debate_date), 
      0, 
      2025 - year(speaker_first_debate_date)
    )
  )

### Delete unnecessary columns

In [148]:
# delete following columns:
data <- data %>% select(-c(tournament_name, person_id, opponent_team_name, speaker_first_debate_date, debate_date))

### Reorganize columns

In [149]:
# Desired order
features <- c("tournament_id", "debate_id", "motion", "speaker_name", "is_male", "aff_side", "speaker_points", "ballots_gained",
              "speaker_position_1", "speaker_position_2", "rolling_avg_5", "avg_teammate_rolling_score",
              "avg_opponent_rolling_score", "cumulative_debates", "years_debating",
              "Environment / Climate", "Social Justice", "Economics", "Philosophy", "War / Conflict", 
              "Arts / Entertainment", "International Relations", "Psychology", "History", "Politics", 
              "Social Media", "Science / Technology", "Family / Parenting", "Urban Planning / Housing", 
              "Law", "Sports", "Religion", "Work / Labor", "Culture", "Media / Journalism")

# Set the desired order
data <- data %>%
  select(all_of(features))

### Save as "processed_data.csv"

In [150]:
# Save
write_csv(data, "data/processed_data.csv")

In [151]:
glimpse(data)

Rows: 3,300
Columns: 35
$ tournament_id              <int> 307, 307, 307, 307, 307, 307, 307, 307, 307…
$ debate_id                  <int> 10700, 10700, 10700, 10700, 10700, 10700, 1…
$ motion                     <chr> "Rozvinuté země by měly kompenzovat rozvojo…
$ speaker_name               <chr> "Ondráčková Zuzana", "Petrencová Nikol", "D…
$ is_male                    <lgl> FALSE, FALSE, FALSE, TRUE, TRUE, TRUE, FALS…
$ aff_side                   <lgl> TRUE, TRUE, TRUE, FALSE, FALSE, FALSE, TRUE…
$ speaker_points             <int> 69, 68, 60, 84, 81, 83, 70, 69, 68, 70, 68,…
$ ballots_gained             <int> 0, 0, 0, 3, 3, 3, 0, 0, 0, 3, 3, 3, 0, 0, 0…
$ speaker_position_1         <lgl> TRUE, FALSE, FALSE, TRUE, FALSE, FALSE, TRU…
$ speaker_position_2         <lgl> FALSE, TRUE, FALSE, FALSE, TRUE, FALSE, FAL…
$ rolling_avg_5              <dbl> NaN, NaN, NaN, NaN, NaN, NaN, NaN, NaN, NaN…
$ avg_teammate_rolling_score <dbl> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA,…
$ avg_opponent_r